# Notebook 03 — Visual Review of Near-Duplicate Candidates

## Purpose

Our first candidate stratified split had **2,631 training**, **465 validation**, and **751 publisher test** images, with 48 classes in all three subsets.

A 64-bit dHash screen found **23 potential publisher train↔test pairs** and **12 potential candidate train↔validation pairs** at Hamming distance ≤3. **These are candidate image PAIRS, not confirmed leaks, and not counts of distinct images.**

We will display each flagged pair side by side, record human judgments, and **keep the current manifest provisional** until the group resolves these findings.

**Do not use test images in this notebook for tuning preprocessing, augmentations, or model hyperparameters.** Viewing test files is restricted here to investigating data integrity and potential split contamination.

## 1. Load the candidate pairs and metadata

The pair CSV was created by Notebook 02 under `splits/pakistan/`. Every path is relative to the repository. This notebook does not modify source images.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageOps

cwd = Path.cwd().resolve()
REPO = next((p for p in (cwd, *cwd.parents) if (p / "configs" / "experiments.yaml").exists()), None)
if REPO is None:
    raise RuntimeError("Open the notebook from inside the vehicle-recognition repository.")

SPLIT_DIR = REPO / "splits" / "pakistan"
PAIR_PATH = SPLIT_DIR / "candidate_near_duplicate_pairs.csv"
META_PATH = SPLIT_DIR / "candidate_split_metadata.json"
if not PAIR_PATH.exists() or not META_PATH.exists():
    raise FileNotFoundError("Missing candidate outputs. Run Notebook 02 through its export cell first.")

pairs = pd.read_csv(PAIR_PATH).fillna("")
meta = json.loads(META_PATH.read_text(encoding="utf-8"))
needed = {"boundary", "distance", "left_path", "right_path", "same_class"}
assert needed.issubset(pairs.columns), f"Missing columns: {needed - set(pairs.columns)}"
pairs = pairs.sort_values(["boundary", "distance", "left_path", "right_path"]).reset_index(drop=True)
pairs.insert(0, "review_id", ["P" + str(i + 1).zfill(3) for i in range(len(pairs))])

print("Candidate split status:", meta["status"])
print("Candidate pairs present in review CSV:", len(pairs))
print("Publisher train↔test flagged PAIRS:", meta["publisher_train_vs_test_candidate_pairs"])
print("Candidate train↔val flagged PAIRS:", meta["candidate_train_vs_val_candidate_pairs"])
print("Boundary counts in saved CSV:")
display(pairs.groupby(["boundary", "distance"]).size().rename("pairs").reset_index())
if len(pairs) < (meta["publisher_train_vs_test_candidate_pairs"] + meta["candidate_train_vs_val_candidate_pairs"]):
    print("WARNING: Notebook 02 saved at most 100 example pairs per boundary; this file is a sample, not a complete review.")
for column in ("left_path", "right_path"):
    missing = [x for x in pairs[column] if not (REPO / x).is_file()]
    if missing:
        raise FileNotFoundError(f"Missing image paths in {column}: {missing[:3]}")

## 2. Display every flagged pair side by side

Each pair uses two columns:
- **Left:** the first image in the pair.
- **Right:** the second image in the pair.

Look for identical photographs (possibly compressed differently), adjacent frames of the *same physical car*, shared distinctive background details, or two merely similar-looking vehicles.

Do **not** assume that two photos are duplicates just because they share a make/model or because dHash distance is small.

The output is displayed directly in the notebook. It does not write any derived vehicle images to GitHub.

In [ ]:
def show_pair_batch(batch, title):
    if batch.empty:
        return
    fig, axes = plt.subplots(len(batch), 2, figsize=(12, 3.1 * len(batch)), squeeze=False)
    for i, row in enumerate(batch.itertuples(index=False)):
        for col, rel in enumerate((row.left_path, row.right_path)):
            ax = axes[i, col]
            with Image.open(REPO / rel) as im:
                im = ImageOps.exif_transpose(im)
                im = im.convert("RGB")
                im.thumbnail((720, 450), Image.Resampling.LANCZOS)
                ax.imshow(im)
            ax.axis("off")
            ax.set_title(f"{row.review_id} | {('A' if col == 0 else 'B')} | {Path(rel).parent.name}", fontsize=9)
        axes[i, 0].set_ylabel(f"dHash={row.distance}", fontsize=9)
        axes[i, 0].text(0, -0.10, Path(row.left_path).name[:65], transform=axes[i, 0].transAxes, fontsize=8)
        axes[i, 1].text(0, -0.10, Path(row.right_path).name[:65], transform=axes[i, 1].transAxes, fontsize=8)
    fig.suptitle(title, fontsize=12, y=1.005)
    plt.tight_layout()
    plt.show()

for boundary in ("publisher_train_vs_test", "candidate_train_vs_val"):
    subset = pairs[pairs["boundary"].eq(boundary)]
    print(f"\n=== {boundary}: {len(subset)} flagged pairs ===")
    for start in range(0, len(subset), 5):
        batch = subset.iloc[start:start + 5]
        show_pair_batch(batch, f"{boundary} — pairs {start + 1}–{start + len(batch)}")

## 3. Record the team's judgments in a CSV

The first run creates `splits/pakistan/candidate_pair_review.csv` with one row for each displayed pair and verdict **unreviewed**. It does **not overwrite** a previously edited review, so you can keep your judgments when rerunning this notebook.

Open the CSV in VS Code or a spreadsheet editor and replace `unreviewed` with one of:

- `same_photo_or_adjacent_frame`: identical photo, alternate encoding/crop, or obvious neighboring frames of the same vehicle in the same scene
- `likely_same_vehicle`: likely the same physical vehicle in distinct images
- `different`: clearly different vehicles/scenes, despite similar hash
- `uncertain`: requires a second reviewer

Enter your initials and brief `notes` for each pair. If the folder labels disagree with visual evidence, mention that in notes. We will review likely leaks before changing **any** image assignment. The publisher's test set is preserved for now.

In [ ]:
review_path = SPLIT_DIR / "candidate_pair_review.csv"
if not review_path.exists():
    review = pairs.copy()
    review["verdict"] = "unreviewed"
    review["reviewer"] = ""
    review["notes"] = ""
    review.to_csv(review_path, index=False)
    print("Created blank human-review template:", review_path)
else:
    print("Found existing review; leaving human annotations untouched:", review_path)

review = pd.read_csv(review_path).fillna("")
assert len(review) == len(pairs), "Review row count differs from the candidate pair count; inspect both files."
print("Current verdict distribution:")
display(review["verdict"].value_counts(dropna=False).rename("pairs").to_frame())
display(review[["review_id", "boundary", "distance", "verdict", "reviewer", "notes"]].head(12))

## 4. Decision rules before approving the split

- **Candidate train vs validation:** Confirmed overlapping photos/same-vehicle video frames must not be split between training and validation. Reassign related *training-origin* samples together or adopt a group-aware split if source groups can be reconstructed; save the rationale and all new manifest hashes.
- **Publisher train vs test:** Preserve the original publisher split for a transparent baseline, and flag any confirmed contamination in the report. If warranted, add a **clearly named, separate sensitivity evaluation** excluding suspect leaked images; don't silently replace the original benchmark.
- **Uncertain pairs:** Obtain a second opinion; do not auto-delete based on a threshold.
- **No flagged pairs:** This still does *not* prove no leakage, since perceptual hashing misses some semantically identical images.

The dataset is **not approved for final training** until these decisions are documented in the experiment log and approved manifest.

## Checkpoint

Share your verdict counts (e.g., how many are `different`, `likely_same_vehicle`, and `uncertain`) and a few difficult pairs. Once the review is complete, we'll fix the final split manifests and then implement the shared PyTorch DataLoader. Avoid committing raw vehicle image previews or large executed notebook outputs.